# **Fine Tunning**  👨🏻‍💻

In [1]:
#Conection with Google Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
dataset_amazon_titles = "/content/drive/MyDrive/DatasetAmazonTitles/trn.json"
amazon_titles_alpaca_format_path = "/content/drive/MyDrive/DatasetAmazonTitles/amazon_titles_alpaca_format.jsonl"

In [3]:
import json
import random

instructions = [
    "Describe a product by title",
    "Describe this product",
    "Give a description of the product type",
    "Explain what this product is",
    "Provide details about this product",
    "Sumarize the type of this product"
    ]

with open(dataset_amazon_titles, "r", encoding="utf-8") as f, open(amazon_titles_alpaca_format_path, "w", encoding="utf-8") as out:
  for line in f:
    data = json.loads(line)
    title = data.get("title")
    content = data.get("content")

    if not title or not content:
      continue

    dict_alpaca_style = {
        "Instruction": random.choice(instructions),
        "Input": title.strip(),
        "Output": content.strip()
    }

    out.write(json.dumps(dict_alpaca_style, ensure_ascii=False) + "\n")


In [4]:
# check the new file

!head -n 10 /content/drive/MyDrive/DatasetAmazonTitles/amazon_titles_alpaca_format.jsonl

{"Instruction": "Describe a product by title", "Input": "Girls Ballet Tutu Neon Pink", "Output": "High quality 3 layer ballet tutu. 12 inches in length"}
{"Instruction": "Explain what this product is", "Input": "Mog's Kittens", "Output": "Judith Kerr&#8217;s best&#8211;selling adventures of that endearing (and exasperating) cat Mog have entertained children for more than 30 years. Now, even infants and toddlers can enjoy meeting this loveable feline. These sturdy little board books&#8212;with their bright, simple pictures, easy text, and hand&#8211;friendly formats&#8212;are just the thing to delight the very young. Ages 6 months&#8211;2 years."}
{"Instruction": "Sumarize the type of this product", "Input": "Girls Ballet Tutu Neon Blue", "Output": "Dance tutu for girls ages 2-8 years. Perfect for dance practice, recitals and performances, costumes or just for fun!"}
{"Instruction": "Describe a product by title", "Input": "The Prophet", "Output": "In a distant, timeless place, a mysteri

In [5]:
!pip install clean-text ftfy langdetect --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 19.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.4/175.4 kB 16.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.0 MB/s eta 0:00:00


In [7]:
import re, html
import ftfy
from cleantext import clean as clean_text
from langdetect import detect

# 4. Funções de pré-processamento
def preprocess(text: str) -> str:
    if not text:
        return ""
    text = ftfy.fix_text(text)            # Corrigir encoding bugado
    text = html.unescape(text)            # Corrigir entidades HTML
    text = clean_text(                    # Limpeza geral
        text,
        fix_unicode=True,
        no_urls=True,
        no_emails=True,
        no_phone_numbers=True,
        no_line_breaks=True,
        lower=False,
    )
    text = re.sub(r"\s+", " ", text)      # Normalizar espaços
    return text.strip()

def clean(example):
    title = preprocess(example["Input"])
    desc  = preprocess(example["Output"])

    # Regras de Input (título)
    if len(title) < 3 or re.match(r"^[0-9\s\-_]+$", title):
        return None

    # Regras de Output (descrição)
    if len(desc) < 10 or len(desc) > 1000:
        return None
    if desc.startswith('"') and desc.endswith('"'):
        return None
    if re.search(r"(BUY NOW|FREE SHIPPING)", desc.upper()):
        return None

    # Linguagem (descartar não-inglês)
    try:
        if detect(desc) != "en":
            return None
    except:
        return None

    return {"Instruction": example["Instruction"], "Input": title, "Output": desc}


In [8]:

from datasets import load_dataset

dataset_path = "/content/drive/MyDrive/DatasetAmazonTitles/amazon_titles_alpaca_format.jsonl"
dataset = load_dataset("json", data_files=dataset_path, split="train")

dataset_clean = dataset.map(clean).filter(lambda x: x is not None)

print("Antes:", len(dataset))
print("Depois da limpeza:", len(dataset_clean))

# Mostrar exemplo antes/depois
idx = 0
print("\n🔹 Exemplo antes:")
print(dataset[idx])
print("\n🔹 Exemplo depois:")
print(dataset_clean[idx])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1390403 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1390403 [00:00<?, ? examples/s]

Antes: 1390403
Depois da limpeza: 1390403

🔹 Exemplo antes:
{'Instruction': 'Describe a product by title', 'Input': 'Girls Ballet Tutu Neon Pink', 'Output': 'High quality 3 layer ballet tutu. 12 inches in length'}

🔹 Exemplo depois:
{'Instruction': 'Describe a product by title', 'Input': 'Girls Ballet Tutu Neon Pink', 'Output': 'High quality 3 layer ballet tutu. 12 inches in length'}


In [15]:
from huggingface_hub import notebook_login
notebook_login()

In [16]:
from datasets import DatasetDict

dataset_split = dataset_clean.train_test_split(test_size=0.1, seed=42)

dataset_split = DatasetDict({
    "train": dataset_split["train"],
    "validation": dataset_split["test"]
})


dataset_split.push_to_hub("guillherms/amazon_titles_alpaca_cleaned_v1")

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/626 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :   0%|          |  524kB /  293MB            

...

Creating parquet from Arrow format:   0%|          | 0/140 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :   6%|5         | 3.68MB / 65.3MB            

CommitInfo(commit_url='https://huggingface.co/datasets/guillherms/amazon_titles_alpaca_cleaned_v1/commit/2cb19cf45afdae9de01aba703e889e2c4c48c399', commit_message='Upload dataset', commit_description='', oid='2cb19cf45afdae9de01aba703e889e2c4c48c399', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/guillherms/amazon_titles_alpaca_cleaned_v1', endpoint='https://huggingface.co', repo_type='dataset', repo_id='guillherms/amazon_titles_alpaca_cleaned_v1'), pr_revision=None, pr_num=None)

Treinando

In [17]:
from datasets import load_dataset

dataset = load_dataset("guillherms/amazon_titles_alpaca_cleaned_v1", streaming=True)
train_dataset = dataset["train"]

for example in train_dataset.take(3):
    print(example)

README.md:   0%|          | 0.00/500 [00:00<?, ?B/s]

{'Instruction': 'Sumarize the type of this product', 'Input': 'Bully Tools 92353 12-Gauge Garden Hoe with Fiberglass Handle', 'Output': 'The Bully Tools 6-1/4-Inch wide Garden Hoe. Features a Heavy Duty Fiberglass 56-Inch tool length Handle with triple wall construction. 12 gauge steel head. Great quality that you expect from all Bully Tools.'}
{'Instruction': 'Describe a product by title', 'Input': 'Samsung Galaxy S3 i9300 SGH-i747 Rubberized Cover - Purple', 'Output': 'This hard cover case is specifically designed for your phone.  It will protect your phone from unwanted scratches. Give your phone an extra edge by using this product.'}
{'Instruction': 'Sumarize the type of this product', 'Input': "The Gang That Couldn't Shoot Straight", 'Output': "Take a glance at the credits and you'll see that director James Goldstone's 1971 comedyThe Gang That Couldn't Shoot Straightis the work of some mighty impressive names. Screenwriter Waldo Salt had already won an Oscar forMidnight Cowboy, an

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformer<0.9.0" peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [ ]:
from unsloth import to_sharegpt
from datasets import load_dataset

dataset_path = "/content/drive/MyDrive/DatasetAmazonTitles/amazon_titles_alpaca_format_v3.jsonl"
dataset = load_dataset("json", data_files=dataset_path, split="train")
print(dataset.column_names)
print(dataset[0])

In [ ]:
from unsloth import to_sharegpt

dataset = to_sharegpt(
    dataset,
    merged_prompt="{Instruction}[[\nYour input is:\n{Input}]]",
    output_column_name="Output",
    conversation_extension=3,  # Select more to handle longer conversations
)

In [ ]:
from unsloth import standardize_sharegpt

dataset = standardize_sharegpt(dataset)

In [ ]:
from unsloth import apply_chat_template

dataset = apply_chat_template(
    dataset,
    tokenizer=tokenizer,
    # default_system_message = "You are a helpful assistant", << [OPTIONAL]
)